# Téléchargement & exploration

possible run en local (aucun gpu require)

Dataset réel : [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) (question + schéma `CREATE TABLE` → requête SQL). Ce notebook prototype avant d'industrialiser dans `src/download_dataset.py`.

In [1]:
import sys
sys.path.append('../src')
import json
import pandas as pd
from datasets import load_dataset
from prompts import build_instruction

## télécharge les données brute & exploit

In [2]:
ds = load_dataset('b-mc2/sql-create-context', split='train')
print(len(ds), ds.column_names)
pd.DataFrame(ds[:5])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

z:\Personal project\AI_ENGINEER\LLM_fine_tuning\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Neo\.cache\huggingface\hub\datasets--b-mc2--sql-create-context. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
z:\Personal project\AI_ENGINEER\LLM_fine_tuning\.venv\Lib\site-packages\huggingface_hub\fi

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

78577 ['answer', 'question', 'context']


,answer,question,context
0,SELECT COUNT(*) FROM head WHERE age > 56,How many heads of the departments are older th...,CREATE TABLE head (age INTEGER)
1,"SELECT name, born_state, age FROM head ORDER B...","List the name, born state and age of the heads...","CREATE TABLE head (name VARCHAR, born_state VA..."
2,"SELECT creation, name, budget_in_billions FROM...","List the creation year, name and budget of eac...","CREATE TABLE department (creation VARCHAR, nam..."
3,"SELECT MAX(budget_in_billions), MIN(budget_in_...",What are the maximum and minimum budget of the...,CREATE TABLE department (budget_in_billions IN...
4,SELECT AVG(num_employees) FROM department WHER...,What is the average number of employees of the...,CREATE TABLE department (num_employees INTEGER...


## Longueurs (pour choisir `max_seq_length`)

Le prompt contient le schéma : vérifier que 512 tokens suffisent (≈ 4 caractères par token).

In [3]:
df = pd.DataFrame(ds.shuffle(seed=42).select(range(2000)))
df['n_chars'] = (df['context'] + df['question'] + df['answer']).str.len()
print(df['n_chars'].describe())
print('Part > 1800 caractères (~450 tokens):', (df['n_chars'] > 1800).mean().round(3))

count    2000.000000
mean      211.459000
std        60.650809
min        95.000000
25%       169.000000
50%       198.000000
75%       236.000000
max       718.000000
Name: n_chars, dtype: float64
Part > 1800 caractères (~450 tokens): 0.0


## Générer train/val avec le script

Sous-échantillon reproductible (seed 42) : 2000 train / 200 val.

In [5]:
!python ../src/download_dataset.py --n_train 2000 --n_val 200 --out ../data/processed

Téléchargement de b-mc2/sql-create-context depuis HuggingFace...
Dataset complet: 78577 exemples, colonnes: ['answer', 'question', 'context']
Train: 2000 exemples -> ..\data\processed\train.jsonl
Val:   200 exemples -> ..\data\processed\val.jsonl


In [6]:
train = [json.loads(l) for l in open('../data/processed/train.jsonl', encoding='utf-8')]
print(train[0]['instruction'])
print('\n--> ', train[0]['output'])

Écris la requête SQL qui répond à la question, en utilisant uniquement le schéma fourni. Réponds uniquement avec la requête SQL.

Schéma:
CREATE TABLE table_name_69 (height VARCHAR, name VARCHAR)

Question: How tall is Dylan Postl?

-->  SELECT height FROM table_name_69 WHERE name = "dylan postl"
